In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasters as rt
from gedi_canopy_height import load_canopy_height
from ECOv002_calval_tables import load_combined_eco_flux_ec_filtered, load_metadata_ebc_filt
from os.path import join
from datetime import datetime, date, time
from dateutil import parser
import rasters as rt
from GEOS5FP import GEOS5FP
from koppengeiger import load_koppen_geiger
from solar_apparent_time import UTC_to_solar
import sun_angles
from gedi_canopy_height import load_canopy_height
from MODISCI import MODISCI
from FLiESANN import FLiESANN
from BESS_JPL import BESS_JPL
from BESS_JPL import load_NDVI_minimum
from BESS_JPL import load_NDVI_maximum
from BESS_JPL import load_C4_fraction
from BESS_JPL import load_carbon_uptake_efficiency
from BESS_JPL import load_kn
from BESS_JPL import load_peakVCmax_C3
from BESS_JPL import load_peakVCmax_C4
from BESS_JPL import load_ball_berry_intercept_C3
from BESS_JPL import load_ball_berry_slope_C3
from BESS_JPL import load_ball_berry_slope_C4
from BESS_JPL import BALL_BERRY_INTERCEPT_C4
from matplotlib.colors import LinearSegmentedColormap
import logging

In [2]:
repo_root = os.path.dirname(os.getcwd())
package_dir = os.path.join(repo_root, 'BESS_JPL')
generated_input_table_filename = os.path.join(package_dir, "ECOv002-static-tower-BESS-JPL-inputs.csv")

In [3]:
tower_locations_df = load_metadata_ebc_filt()
tower_locations_df

,Site ID,Name,Lat,Long,Elev,Clim,Veg,MAT,MAP,StartDate,EndDate,LE_count,closure_ratio,geometry
0,US-NC3,NC_Clearcut#3,35.7990,-76.6560,5.0,Cfa,ENF,16.6,1320.00,10/1/18 05:00,1/1/22 05:00,9576,1.02,POINT (-76.656 35.799)
1,PE-QFR,Quistococha Forest Reserve,-3.8344,-73.3190,104.0,Af,WET,NaN,NaN,10/1/18 05:00,1/1/20 05:00,6859,0.98,POINT (-73.319 -3.8344)
2,US-Mi3,LTAR UCB (Upper Chesapeake Bay) Miscanthus 3,41.8222,-80.6370,270.0,Dfb,CVM,10.5,1012.70,10/1/18 05:00,12/28/19 04:00,12170,0.92,POINT (-80.637 41.8222)
3,US-NC4,NC_AlligatorRiver,35.7879,-75.9038,1.0,Cfa,WET,16.6,1311.00,10/1/18 05:00,1/1/22 05:00,20890,0.90,POINT (-75.9038 35.7879)
4,CA-DB2,Delta Burns Bog 2,49.1190,-122.9951,4.0,NaN,WET,NaN,NaN,1/1/19 08:30,1/1/21 08:00,11884,0.89,POINT (-122.9951 49.119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,US-xSL,"NEON North Sterling, CO (STER)",40.4619,-103.0293,1364.0,Bsk,CRO,9.7,432.67,10/1/18 07:00,12/1/22 00:00,23176,0.60,POINT (-103.0293 40.4619)
117,US-xWD,NEON Woodworth (WOOD),47.1282,-99.2414,579.0,Dfb,GRA,4.9,493.76,10/1/18 06:00,12/1/22 00:00,20494,0.60,POINT (-99.2414 47.1282)
118,US-CS4,Central Sands Irrigated Agricultural Field,44.1597,-89.5475,328.0,Dfa,CRO,7.0,830.00,1/1/20 06:30,4/27/21 06:00,6359,0.60,POINT (-89.5475 44.1597)
119,US-xAE,NEON Klemme Range Research Station (OAES),35.4106,-99.0588,516.0,Cfa,GRA,15.5,778.85,10/1/18 06:00,12/1/22 00:00,29615,0.60,POINT (-99.0588 35.4106)


In [4]:
tower_IDs = list(tower_locations_df["Site ID"])
tower_IDs

['US-NC3',
 'PE-QFR',
 'US-Mi3',
 'US-NC4',
 'CA-DB2',
 'US-Sne',
 'US-Mi1',
 'US-PFe',
 'US-NR1',
 'US-Vcp',
 'US-xAB',
 'US-HBK',
 'US-EDN',
 'US-PFh',
 'US-Me6',
 'US-NC2',
 'US-Whs',
 'US-SRM',
 'US-CS1',
 'US-PFs',
 'US-PFg',
 'US-Ha1',
 'US-Ne2',
 'US-PFm',
 'US-PFr',
 'US-PFL',
 'US-PFj',
 'US-CC2',
 'US-NR3',
 'US-PFt',
 'CA-Cbo',
 'PR-xLA',
 'US-Ne3',
 'US-PHM',
 'US-Tw1',
 'US-Hn3',
 'US-PFk',
 'US-CS2',
 'US-ONA',
 'US-CC1',
 'US-Rls',
 'US-UMd',
 'US-PFq',
 'US-PFn',
 'US-HB3',
 'US-DFC',
 'US-Wkg',
 'US-CF1',
 'US-Ho2',
 'US-Snf',
 'US-Me2',
 'US-CF3',
 'US-Ne1',
 'US-Ha2',
 'CA-Ca3',
 'US-SP1',
 'US-UMB',
 'US-Bi1',
 'US-PFi',
 'US-CF2',
 'US-KFS',
 'US-CMW',
 'US-Ro4',
 'US-PAS',
 'US-xYE',
 'US-Los',
 'US-PFd',
 'US-KM4',
 'US-Syv',
 'US-WCr',
 'US-Rws',
 'US-xJE',
 'US-SRG',
 'US-xNW',
 'US-Rms',
 'US-NR4',
 'US-xDS',
 'US-xJR',
 'US-Bar',
 'US-ALQ',
 'US-MMS',
 'US-UC1',
 'US-PFb',
 'US-xDL',
 'US-xPU',
 'US-xBL',
 'US-xST',
 'US-xSB',
 'US-xRN',
 'US-UC2',
 'US-xTR',

In [5]:
tower_names = list(tower_locations_df.Name)
tower_names

['NC_Clearcut#3',
 'Quistococha Forest Reserve',
 'LTAR UCB (Upper Chesapeake Bay) Miscanthus 3',
 'NC_AlligatorRiver',
 'Delta Burns Bog 2',
 'Sherman Island Restored Wetland',
 'LTAR UCB (Upper Chesapeake Bay) Miscanthus 1',
 'NW4 Lake-1 CHEESEHEAD 2019',
 'Niwot Ridge Forest (LTER NWT1)',
 'Valles Caldera Ponderosa Pine',
 'NEON Abby Road (ABBY)',
 'Hubbard Brook Experimental Forest',
 'Eden Landing Ecological Reserve',
 'NE2 Pine-3 CHEESEHEAD 2019',
 'Metolius Young Pine Burn',
 'NC_Loblolly Plantation',
 'Walnut Gulch Lucky Hills Shrub',
 'Santa Rita Mesquite',
 'Central Sands Irrigated Agricultural Field',
 'SE5 Aspen-5 CHEESEHEAD 2019',
 'NE1 Pine-2 CHEESEHEAD 2019',
 'Harvard Forest EMS Tower (HFR1)',
 'Mead - irrigated maize-soybean rotation site',
 'SW3 Hardwood-2 CHEESEHEAD 2019',
 'SE4 Tussock-2 CHEESEHEAD 2019',
 'SW2 Aspen-3 CHEESEHEAD 2019',
 'NE4 Maple-1 CHEESEHEAD 2019',
 'Coloma Corn 2',
 'Niwot Ridge Alpine (T-Van West)',
 'SE6 Pine-4 CHEESEHEAD 2019',
 'Ontario - Mi

In [6]:
tower_data_df = load_combined_eco_flux_ec_filtered()
tower_data_df

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,ESIrn_STIC,ESIrn_PTJPLSM,ESIrn_MOD16,ESIrn_BESS,ESIrn_Unc_ECO,ESIrn_LEcorr50,JET,eco_time_utc,Site Name,Date-Time
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0.686404,0.779526,0.997448,0.199396,0.301927,0.737734,288.683585,2019-10-02 19:09:40,US-NC3,2019-10-02 19:09:40
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0.360762,0.582912,0.994784,0.356192,0.260957,0.413558,303.615450,2019-06-23 18:17:17,US-Mi3,2019-06-23 18:17:17
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0.566752,0.452768,0.995059,0.533156,0.211423,0.558382,345.793640,2019-06-27 16:35:42,US-Mi3,2019-06-27 16:35:42
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0.531036,0.401006,0.995685,0.521065,0.225106,0.501352,329.812600,2019-06-30 15:44:10,US-Mi3,2019-06-30 15:44:10
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0.559024,0.445342,0.996002,0.462290,0.223742,0.375202,262.035285,2019-07-01 14:53:48,US-Mi3,2019-07-01 14:53:48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0.434453,0.093619,0.500132,1.055910,0.345397,0.054837,76.284270,2021-12-11 16:01:12,US-xAE,2021-12-11 16:01:12
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0.701256,0.133500,0.393938,0.732986,0.243751,0.264294,91.006255,2022-03-25 22:45:31,US-xAE,2022-03-25 22:45:31
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0.550616,0.234624,0.503546,0.000000,0.222162,0.355559,87.060413,2022-04-12 22:53:09,US-xAE,2022-04-12 22:53:09
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0.010924,0.163120,0.510292,0.566350,0.233056,0.269973,83.462274,2022-04-14 14:45:37,US-xAE,2022-04-14 14:45:37


In [7]:
tower_data_df.columns

Index(['Unnamed: 0', 'ID', 'vegetation', 'climate', 'STICinst', 'BESSinst',
       'MOD16inst', 'PTJPLSMinst', 'ETinst', 'ETinstUncertainty', 'PET', 'Rn',
       'ESI', 'RH', 'Ta', 'LST', 'SM', 'NDVI', 'NDVI-UQ', 'albedo',
       'albedo-UQ', 'LST_err', 'view_zenith', 'Rg', 'EmisWB', 'time_utc',
       'solar_time', 'solar_hour', 'local_time', 'LE', 'LE_filt', 'LEcorr25',
       'LEcorr50', 'LEcorr75', 'LEcorr_ann', 'H_filt', 'Hcorr25', 'Hcorr50',
       'Hcorr75', 'Hcorr_ann', 'NETRAD_filt', 'G_filt', 'SM_surf', 'SM_rz',
       'AirTempC', 'SW_IN', 'RH_percentage', 'ESIrn_STIC', 'ESIrn_PTJPLSM',
       'ESIrn_MOD16', 'ESIrn_BESS', 'ESIrn_Unc_ECO', 'ESIrn_LEcorr50', 'JET',
       'eco_time_utc', 'Site Name', 'Date-Time'],
      dtype='object')

In [8]:
# Create MultiPoint geometry for tower locations
tower_points = rt.MultiPoint(
    x=tower_locations_df['Long'].values,
    y=tower_locations_df['Lat'].values
)

tower_points

MULTIPOINT ((-76.656 35.799), (-73.319 -3.8344), (-80.637 41.8222), (-75.9038 35.7879), (-122.9951 49.119), (-121.7547 38.0369), (-80.6313 41.7727), (-90.3004 45.9793), (-105.5464 40.0329), (-106.5974 35.8624), (-122.3303 45.7624), (-71.7181 43.9397), (-122.114 37.6156), (-90.2406 45.9557), (-121.6078 44.3233), (-76.6685 35.803), (-110.0522 31.7438), (-110.8661 31.8214), (-89.5379 44.1031), (-90.2382 45.9381), (-90.2723 45.9735), (-72.1715 42.5378), (-96.4701 41.1649), (-90.3099 45.9207), (-90.2475 45.9245), (-90.3177 45.9409), (-90.227 45.9619), (-89.6196 44.1039), (-105.5864 40.052), (-90.2288 45.9197), (-79.9333 44.3167), (-67.0769 18.0212), (-96.4397 41.1797), (-70.8301 42.7423), (-121.6469 38.1074), (-119.4614 46.6878), (-90.3425 45.9149), (-89.5002 44.1467), (-81.9509 27.3836), (-89.6787 44.0732), (-116.7356 43.1439), (-84.6975 45.5625), (-90.2475 45.9271), (-90.2823 45.9392), (-79.2322 33.3482), (-89.7117 43.3448), (-109.9419 31.7365), (-117.0821 46.7815), (-68.747 45.2091), (-1

In [9]:
geometry = tower_points

In [10]:
type(geometry)

rasters.multi_point.MultiPoint

In [11]:
NDVI_minimum = load_NDVI_minimum(geometry=geometry)
NDVI_minimum

array([ 0.40873316,  0.65735943,  0.02790972,  0.59135757,  0.39971167,
        0.23912057,  0.01333595,  0.1078617 ,  0.16296521,  0.18605683,
        0.56599368,  0.14425548,  0.18765256,  0.12093302,  0.24290999,
        0.43705223,  0.13977007,  0.16875774,  0.01098392,  0.12107238,
        0.13489336,  0.1799916 , -0.02101911,  0.10283321,  0.1178903 ,
        0.1024973 ,  0.1264813 , -0.00343339,  0.08074605,  0.12780713,
        0.06329029,  0.44800568, -0.02162073,  0.00205075,  0.27353726,
       -0.0130249 ,  0.10267654,  0.01184704,  0.48596438,  0.01410842,
       -0.01721634,  0.0184568 ,  0.11820108,  0.11214952,  0.49291273,
       -0.00731678,  0.14524988, -0.02447569,  0.218314  ,  0.27975733,
        0.26164138, -0.01203388, -0.02100921,  0.18249513,  0.35407636,
        0.56918121,  0.03174983,  0.2691256 ,  0.13597418, -0.02196958,
        0.27267856,  0.14778325, -0.02155501,  0.48697821,  0.00785267,
        0.08206635,  0.10728203,  0.00824275,  0.0909669 ,  0.07

In [12]:
NDVI_maximum = load_NDVI_maximum(geometry=geometry)
NDVI_maximum

array([0.8556929 , 0.82660494, 0.85546051, 0.86975765, 0.67573063,
       0.55724579, 0.84100305, 0.87923122, 0.67567462, 0.69142193,
       0.86253765, 0.90478519, 0.4185708 , 0.87975512, 0.76952894,
       0.85392176, 0.37919385, 0.4243245 , 0.7909789 , 0.87431379,
       0.88419755, 0.9038529 , 0.83341199, 0.87900961, 0.87514362,
       0.8778957 , 0.88260898, 0.82363582, 0.57935926, 0.8708631 ,
       0.82487249, 0.74840622, 0.81946942, 0.7401394 , 0.70919145,
       0.44533253, 0.8758384 , 0.78626713, 0.75425117, 0.87121866,
       0.48672659, 0.82310268, 0.87524849, 0.88099706, 0.76964672,
       0.81410135, 0.47096672, 0.85596113, 0.8721062 , 0.60724544,
       0.81240164, 0.81819442, 0.83378512, 0.90236052, 0.88792623,
       0.79402437, 0.82487303, 0.74774593, 0.88698888, 0.84864159,
       0.79867407, 0.46289993, 0.82892517, 0.75627672, 0.60320478,
       0.85685256, 0.8788174 , 0.83286317, 0.87849629, 0.89125694,
       0.41941068, 0.79275685, 0.57994892, 0.58362092, 0.59386

In [13]:
C4_fraction = load_C4_fraction(geometry=geometry)
C4_fraction

array([7.75317136e-02, 5.49260800e-04, 3.50451879e-02, 4.62194359e-02,
       3.55873514e-04, 5.16687902e-03, 3.15875861e-02, 4.69371136e-02,
       1.55606723e-02, 7.81100362e-04, 1.66224623e-03, 0.00000000e+00,
       1.18293722e-03, 4.68580356e-02, 3.23919959e-04, 7.94858677e-02,
       5.38372480e-01, 6.74624390e-01, 8.67678880e-02, 4.69671658e-02,
       4.68406221e-02, 3.68686381e-04, 4.41500589e-01, 4.71107899e-02,
       4.70302449e-02, 4.71660958e-02, 4.67580060e-02, 8.65172618e-02,
       1.49460462e-02, 4.70331756e-02, 7.64033193e-03, 0.00000000e+00,
       4.42448067e-01, 0.00000000e+00, 6.10323702e-03, 5.56087140e-03,
       4.71529234e-02, 9.10350446e-02, 7.01153400e-03, 8.30079717e-02,
       4.18958671e-04, 8.91574688e-02, 4.70239868e-02, 4.70600530e-02,
       1.80595238e-02, 4.58021609e-02, 4.86889255e-01, 4.77017623e-03,
       1.29824565e-04, 5.26002449e-03, 3.04685792e-04, 5.13066209e-03,
       4.41846701e-01, 3.64537640e-04, 5.40221955e-05, 7.93627940e-03,
      

In [14]:
carbon_uptake_efficiency = load_carbon_uptake_efficiency(geometry=geometry)
carbon_uptake_efficiency

array([0.08      , 0.06034713, 0.08      , 0.08      , 0.08      ,
       0.08094726, 0.08      , 0.08      , 0.08      , 0.08767799,
       0.08      , 0.08      , 0.08      , 0.08      , 0.08      ,
       0.08      , 0.08521522, 0.08047069, 0.08      , 0.08      ,
       0.08      , 0.08      , 0.08      , 0.08      , 0.08      ,
       0.08      , 0.08      , 0.08      , 0.08276227, 0.08      ,
       0.08057791, 0.08003994, 0.08      , 0.08      , 0.08121726,
       0.09      , 0.08      , 0.08      , 0.08      , 0.08      ,
       0.09      , 0.08053803, 0.08      , 0.08      , 0.08      ,
       0.08      , 0.08942724, 0.08      , 0.08      , 0.08118396,
       0.08      , 0.08      , 0.08      , 0.08      , 0.08      ,
       0.08      , 0.08      , 0.08034583, 0.08      , 0.08      ,
       0.08      , 0.08951679, 0.08      , 0.08      , 0.08941946,
       0.08      , 0.08      , 0.08      , 0.08      , 0.08      ,
       0.09      , 0.08      , 0.08133262, 0.08234185, 0.09   

In [15]:
kn = load_kn(geometry=geometry)
kn

array([0.41      , 0.12503339, 0.41      , 0.41      , 0.41      ,
       0.43841783, 0.41      , 0.41      , 0.41      , 0.64033948,
       0.41      , 0.41      , 0.41      , 0.41      , 0.41      ,
       0.41      , 0.56645664, 0.4241207 , 0.41      , 0.41      ,
       0.41      , 0.41      , 0.41      , 0.41      , 0.41      ,
       0.41      , 0.41      , 0.41      , 0.49286811, 0.41      ,
       0.42733741, 0.4111982 , 0.41      , 0.41      , 0.44651792,
       0.70999998, 0.41      , 0.41      , 0.41      , 0.41      ,
       0.70999998, 0.42614109, 0.41      , 0.41      , 0.41      ,
       0.41      , 0.69281721, 0.41      , 0.41      , 0.44551886,
       0.41      , 0.41      , 0.41      , 0.41      , 0.41      ,
       0.41      , 0.41      , 0.42037485, 0.41      , 0.41      ,
       0.41      , 0.69550353, 0.41      , 0.41      , 0.69258355,
       0.41      , 0.41      , 0.41      , 0.41      , 0.41      ,
       0.70999998, 0.41      , 0.44997853, 0.48025556, 0.70999

In [16]:
peakVCmax_C3 = load_peakVCmax_C3(geometry=geometry)
peakVCmax_C3

array([ 87.34543335,  41.3644872 , 119.43544274,  64.72016472,
       109.98639536,  60.07227719, 120.        ,  68.82643069,
       120.        , 136.89156294,  63.        ,  96.        ,
        56.98456487,  83.22012914, 113.49030088,  85.58671584,
        70.34435465,  62.75310397, 106.01582914,  74.2996212 ,
        62.8809708 ,  92.68174367, 101.        ,  68.75721378,
        72.00066865,  70.54082849,  90.76111754, 104.34782995,
       126.07699569,  65.96400082, 120.51203013,  99.32095752,
       101.        , 107.83236255,  98.2002923 ,  78.        ,
        77.96040449, 102.29944267, 120.        , 119.70852196,
       107.33151698,  95.42972888,  71.88368288,  67.22195801,
        67.956182  , 102.15161504,  77.08358571, 101.        ,
        62.5       ,  50.99845323,  71.10432068,  98.0394211 ,
       101.        ,  91.48117954,  60.64153235, 108.40484293,
        95.77499185, 100.20459451,  94.78676387, 101.        ,
       117.95860351,  77.22685598, 101.        , 120.  

In [17]:
peakVCmax_C4 = load_peakVCmax_C4(geometry=geometry)
peakVCmax_C4

array([ 51.04062441,  41.3644872 , 119.43544274,  64.72016472,
       109.98639536,  49.01803092, 120.        ,  68.82643069,
       120.        ,  58.57613477,  63.        ,  96.        ,
        56.98456487,  83.22012914, 113.49030088,  59.56049992,
        50.52651236,  60.96448204,  58.91125359,  74.2996212 ,
        62.8809708 ,  92.68174367,  37.        ,  68.75721378,
        72.00066865,  70.54082849,  90.76111754,  51.62473084,
        97.90183386,  65.96400082, 112.05939904,  37.47416733,
        37.        , 107.83236255,  37.36517926,  40.        ,
        77.96040449,  42.67651272, 120.        , 118.96186331,
        40.        ,  89.94175752,  71.88368288,  67.22195801,
        67.956182  ,  55.85384456,  41.26006965,  37.        ,
        62.5       ,  35.08656504,  71.10432068,  38.96090291,
        37.        ,  91.48117954,  60.64153235, 108.40484293,
        95.77499185,  37.10374854,  94.78676387,  37.        ,
       111.08232061,  41.06307302,  37.        , 120.  

In [18]:
ball_berry_slope_C3 = load_ball_berry_slope_C3(geometry=geometry)
ball_berry_slope_C3

array([9.5       , 9.5       , 7.5       , 9.5       , 9.5       ,
       9.5       , 7.5       , 7.5       , 7.5       , 7.5       ,
       9.5       , 7.5       , 9.5       , 7.5       , 7.5       ,
       9.5       , 9.5       , 9.5       , 7.5       , 7.5       ,
       7.5       , 7.5       , 7.5       , 7.5       , 7.5       ,
       7.5       , 7.5       , 7.5       , 7.5       , 7.5       ,
       7.5       , 9.5       , 7.5       , 7.5       , 9.5       ,
       9.5       , 7.5       , 7.5       , 9.5       , 7.5       ,
       8.58339009, 7.5       , 7.5       , 7.5       , 9.5       ,
       7.5       , 9.5       , 7.5       , 7.5       , 9.5       ,
       7.5       , 7.5       , 7.5       , 7.5       , 9.39223178,
       9.5       , 7.5       , 9.5       , 7.5       , 7.5       ,
       7.5       , 9.5       , 7.5       , 9.5       , 7.5       ,
       7.5       , 7.5       , 7.5       , 7.5       , 7.5       ,
       9.40520006, 9.5       , 9.5       , 7.5       , 7.78216

In [19]:
ball_berry_slope_C4 = load_ball_berry_slope_C4(geometry=geometry)
ball_berry_slope_C4

array([4.9519627 , 5.        , 4.1211708 , 4.96527226, 4.55028556,
       5.07578091, 4.0999999 , 5.        , 4.0999999 , 5.40525726,
       5.        , 5.        , 5.        , 4.91962964, 4.20278464,
       4.88249207, 5.41721783, 5.03765521, 4.76240807, 4.89980465,
       4.9986334 , 5.        , 5.        , 4.95285164, 4.90527293,
       4.95559023, 4.9399171 , 4.84141856, 4.56958601, 4.96295705,
       5.04623311, 4.9920181 , 5.        , 4.21775125, 5.09738116,
       5.80000019, 4.91736069, 4.93844745, 5.        , 4.11311781,
       5.80000019, 4.92613104, 4.90844516, 4.97306499, 4.77322641,
       5.        , 5.75417947, 5.        , 5.        , 5.09471699,
       4.87203703, 5.        , 5.        , 5.        , 5.        ,
       4.27992477, 4.82307453, 5.02766628, 4.98503487, 5.        ,
       5.        , 5.76134298, 5.        , 5.        , 5.72328074,
       4.91066092, 5.        , 4.49779588, 5.        , 4.74524569,
       5.80000019, 4.25598986, 5.10660945, 4.49811487, 5.80000

In [20]:
ball_berry_intercept_C3 = load_ball_berry_intercept_C3(geometry=geometry)
ball_berry_intercept_C3

array([0.00526687, 0.005     , 0.00988238, 0.00519293, 0.00749841,
       0.00594726, 0.01      , 0.005     , 0.01      , 0.01383899,
       0.005     , 0.005     , 0.005     , 0.0054465 , 0.00942897,
       0.00565282, 0.01021522, 0.00547069, 0.00631995, 0.00555664,
       0.00500759, 0.005     , 0.005     , 0.00526194, 0.00552626,
       0.00524672, 0.00533379, 0.00588101, 0.01138114, 0.00520579,
       0.00557791, 0.00510204, 0.005     , 0.00934583, 0.00621726,
       0.015     , 0.00545911, 0.00534196, 0.005     , 0.00992712,
       0.015     , 0.00618755, 0.00550864, 0.00514964, 0.00625985,
       0.005     , 0.01442724, 0.005     , 0.005     , 0.00618396,
       0.00571091, 0.005     , 0.005     , 0.005     , 0.005     ,
       0.00900042, 0.00598292, 0.00534583, 0.00508314, 0.005     ,
       0.005     , 0.01451678, 0.005     , 0.005     , 0.01458765,
       0.00549633, 0.005     , 0.00779002, 0.005     , 0.0064153 ,
       0.015     , 0.00913339, 0.00633262, 0.01117093, 0.015  

In [21]:
KG_climate = load_koppen_geiger(geometry=geometry)
KG_climate

array([3, 1, 4, 3, 3, 3, 4, 4, 4, 4, 3, 4, 3, 4, 4, 3, 2, 2, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 5, 4, 4, 1, 4, 4, 3, 2, 4, 4, 3, 4, 2, 4, 4, 4,
       3, 4, 2, 4, 4, 3, 4, 4, 4, 4, 3, 3, 4, 3, 4, 4, 4, 2, 4, 3, 4, 4,
       4, 4, 4, 4, 2, 3, 2, 4, 4, 5, 3, 2, 4, 4, 4, 4, 4, 3, 3, 4, 4, 3,
       3, 4, 4, 4, 3, 3, 2, 4, 2, 3, 4, 4, 4, 4, 3, 3, 2, 4, 4, 4, 4, 4,
       4, 4, 1, 3, 4, 3, 2, 4, 4, 3, 4])

In [22]:
from MODISCI import MODISCI

MODISCI_connection = MODISCI()
MODISCI_connection.download()

[2025-09-09 16:06:54 WARNING] netrc credentials not found for daac.ornl.gov
[2025-09-09 16:06:54 INFO] file already downloaded: ~/data/MODISCI/global_clumping_index.tif


'~/data/MODISCI/global_clumping_index.tif'

In [23]:
from MODISCI import MODISCI

MODISCI_connection = MODISCI()

CI = MODISCI_connection.CI(geometry=geometry)
CI

[2025-09-09 16:06:54 WARNING] netrc credentials not found for daac.ornl.gov
[2025-09-09 16:06:54 INFO] file already downloaded: ~/data/MODISCI/global_clumping_index.tif
[2025-09-09 16:06:55 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:55 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:55 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:55 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:55 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:55 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:56 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:56 WARNING] CPLE_NotSupported in driver GTiff does not support open option FILL
[2025-09-09 16:06:56 WARNING] CPLE_NotSupported

array([0.28235294, 0.25490196, 0.28627451, 0.20784314, 0.26666667,
       1.        , 0.29019608, 0.21568627, 0.18431373, 0.2       ,
       0.18823529, 0.26666667, 1.        , 0.2627451 , 0.24705882,
       0.2627451 , 0.28235294, 0.23529412, 0.22352941, 0.25882353,
       0.2627451 , 0.2745098 , 0.29019608, 0.26666667, 0.2627451 ,
       0.2627451 , 0.26666667, 0.26666667, 0.21960784, 0.27058824,
       1.        , 0.30588235, 0.29803922, 1.        , 0.29019608,
       0.23921569, 0.27843137, 0.22745098, 0.28627451, 0.28235294,
       0.28627451, 1.        , 0.25490196, 0.21176471, 0.20392157,
       0.28627451, 0.23529412, 0.30588235, 0.25098039, 1.        ,
       0.19215686, 0.30588235, 0.29019608, 0.26666667, 0.19607843,
       0.20784314, 1.        , 0.28627451, 0.27058824, 0.30588235,
       0.28235294, 0.29411765, 0.2745098 , 0.22745098, 0.22352941,
       0.25882353, 0.20392157, 0.28235294, 0.2627451 , 0.27843137,
       0.28627451, 0.21960784, 0.29803922, 0.22352941, 0.21176

In [24]:
canopy_height_meters = load_canopy_height(geometry=geometry)
canopy_height_meters

array([2.06429022e+01, 2.21400211e+01, 0.00000000e+00, 1.41648272e+01,
       9.91902946e+00, 4.69379179e-01, 0.00000000e+00, 6.83500361e+00,
       1.79422789e+01, 1.57378972e+01, 6.24472538e+00, 2.25277669e+01,
       0.00000000e+00, 1.52666221e+01, 9.92549467e+00, 1.07070754e+01,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 6.99144419e+00,
       1.79398725e+01, 2.22612355e+01, 0.00000000e+00, 1.71406083e+01,
       6.69737404e-02, 1.91192940e+01, 1.10412273e+01, 0.00000000e+00,
       0.00000000e+00, 1.39596213e+01, 1.46374506e+01, 9.72189971e-02,
       0.00000000e+00, 0.00000000e+00, 2.20106021e-02, 0.00000000e+00,
       1.35347424e+01, 1.75170853e+01, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 2.08070523e+01, 1.96582386e+01, 1.67691536e+01,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       1.58615822e+01, 0.00000000e+00, 1.72034385e+01, 0.00000000e+00,
       0.00000000e+00, 2.08270213e+01, 1.32304602e+01, 1.76799707e+01,
      

In [28]:
tower_static_data_gdf = gpd.GeoDataFrame({
    "ID": tower_IDs,
    "name": tower_names,
    "NDVI_minimum": NDVI_minimum,
    "NDVI_maximum": NDVI_maximum,
    "C4_fraction": C4_fraction,
    "carbon_uptake_efficiency": carbon_uptake_efficiency,
    "kn": kn,
    "peakVCmax_C3": peakVCmax_C3,
    "peakVCmax_C4": peakVCmax_C4,
    "ball_berry_slope_C3": ball_berry_slope_C3,
    "ball_berry_slope_C4": ball_berry_slope_C4,
    "ball_berry_intercept_C3": ball_berry_intercept_C3,
    "KG_climate": KG_climate,
    "CI": CI,
    "canopy_height_meters": canopy_height_meters,
    "COT": 0,
    "AOT": 0,
    "Ca": 400,
    "geometry": tower_points
})

tower_static_data_gdf

,ID,name,NDVI_minimum,NDVI_maximum,C4_fraction,carbon_uptake_efficiency,kn,peakVCmax_C3,peakVCmax_C4,ball_berry_slope_C3,ball_berry_slope_C4,ball_berry_intercept_C3,KG_climate,CI,canopy_height_meters,COT,AOT,Ca,geometry
0,US-NC3,NC_Clearcut#3,0.408733,0.855693,0.077532,0.080000,0.410000,87.345433,51.040624,9.5,4.951963,0.005267,3,0.282353,20.642902,0,0,400,POINT (-76.656 35.799)
1,PE-QFR,Quistococha Forest Reserve,0.657359,0.826605,0.000549,0.060347,0.125033,41.364487,41.364487,9.5,5.000000,0.005000,1,0.254902,22.140021,0,0,400,POINT (-73.319 -3.8344)
2,US-Mi3,LTAR UCB (Upper Chesapeake Bay) Miscanthus 3,0.027910,0.855461,0.035045,0.080000,0.410000,119.435443,119.435443,7.5,4.121171,0.009882,4,0.286275,0.000000,0,0,400,POINT (-80.637 41.8222)
3,US-NC4,NC_AlligatorRiver,0.591358,0.869758,0.046219,0.080000,0.410000,64.720165,64.720165,9.5,4.965272,0.005193,3,0.207843,14.164827,0,0,400,POINT (-75.9038 35.7879)
4,CA-DB2,Delta Burns Bog 2,0.399712,0.675731,0.000356,0.080000,0.410000,109.986395,109.986395,9.5,4.550286,0.007498,3,0.266667,9.919029,0,0,400,POINT (-122.9951 49.119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,US-xSL,"NEON North Sterling, CO (STER)",-0.025449,0.495206,0.340776,0.090000,0.710000,78.000000,40.000000,9.5,5.800000,0.015000,2,0.298039,0.000000,0,0,400,POINT (-103.0293 40.4619)
117,US-xWD,NEON Woodworth (WOOD),-0.041785,0.753863,0.032479,0.080000,0.410000,101.000000,37.000000,7.5,5.000000,0.005000,4,0.294118,0.000000,0,0,400,POINT (-99.2414 47.1282)
118,US-CS4,Central Sands Irrigated Agricultural Field,-0.003026,0.776740,0.092454,0.080000,0.410000,101.000000,37.000000,7.5,5.000000,0.005000,4,0.278431,0.000000,0,0,400,POINT (-89.5475 44.1597)
119,US-xAE,NEON Klemme Range Research Station (OAES),0.233503,0.554538,0.371127,0.090000,0.710000,78.000000,40.000000,9.5,5.800000,0.015000,3,0.301961,0.000000,0,0,400,POINT (-99.0588 35.4106)


In [29]:
tower_static_data_gdf.to_csv(generated_input_table_filename, index=False)